In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [2]:
def ndcg_at_k(y_true, y_pred, k=22):
    """nDCG@Kを計算する関数"""
    def dcg(scores):
        return np.sum((2**scores - 1) / np.log2(np.arange(2, scores.size + 2)))

    k = min(k, len(y_true), len(y_pred))  # kをy_trueとy_predの長さ以内に制限

    if k == 0:
        return 0  # kが0の場合はnDCGを0として扱う
    
    # y_trueに基づいて上位k個のインデックスを取得
    order = np.argsort(y_pred)[::-1][:k]
    ideal_order = np.argsort(y_true)[::-1][:k]

    # 範囲外のインデックスを除外
    order = order[order < len(y_true)]
    ideal_order = ideal_order[ideal_order < len(y_true)]
    
    return dcg(y_true[order]) / (dcg(y_true[ideal_order]) + 1e-10)

In [3]:
train_df = pd.read_csv('../001.preprocess/train/train_A.csv')
test_df = pd.read_csv('../001.preprocess/test/test_A.csv')

In [4]:
# カテゴリ変数のエンコード
le_user = LabelEncoder()
le_product = LabelEncoder()
train_df["user_id"] = le_user.fit_transform(train_df["user_id"])
train_df["product_id"] = le_product.fit_transform(train_df["product_id"])

test_df.loc[:, "user_id"] = le_user.transform(test_df["user_id"])

In [5]:
#人気順の最大値を31にする
train_df["rank"] = train_df["rank"].clip(upper=31)

In [6]:
# 特徴量とターゲットの設定
X = train_df[["user_id", "product_id"]]
y = train_df["rank"]

In [7]:
# データの分割
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

In [8]:
# クエリ情報の作成（例：1ユーザーごとの商品の数をクエリサイズとして設定）
train_df["query"] = train_df.groupby("user_id")["product_id"].transform("count")
test_df["query"] = test_df.groupby("user_id")["product_id"].transform("count")

KeyError: 'Column not found: product_id'

In [ ]:
# LightGBM用データフォーマットに変換（query情報を含む）
train_data = lgb.Dataset(X_train, label=y_train, group=train_df.groupby("user_id")["product_id"].transform("count").values)
valid_data = lgb.Dataset(X_valid, label=y_valid, group=test_df.groupby("user_id")["product_id"].transform("count").values)

In [ ]:
# ハイパーパラメータ設定
params = {
    "objective": "lambdarank",  # ランキング問題用
    "metric": "ndcg",  # nDCGを評価指標に設定
    "max_depth": 6,
    "learning_rate": 0.1,
    "lambda_l1": 1.0,  # L1正則化
    "subsample": 0.8,  # データのサンプリング
    "colsample_bytree": 0.8,  # 特徴量のサンプリング
    "min_data_in_leaf": 20,  # リーフの最小データ数
    "early_stopping_rounds": 10,
}

In [ ]:
# モデルの学習
evals_result = {}
bst = lgb.train(params, train_data, num_boost_round=100, valid_sets=[valid_data])

In [ ]:
# 評価結果の取得
evals_result = bst.eval_valid()
print(evals_result)

In [ ]:
# 予測の生成
all_product_ids = train_df["product_id"].unique()

In [ ]:
def generate_predictions_for_user(user):
    user_data = pd.DataFrame({"user_id": [user] * len(all_product_ids), "product_id": all_product_ids})
    user_preds = bst.predict(user_data)
    user_pred_df = pd.DataFrame({"user_id": user, "product_id": all_product_ids, "pred_score": user_preds})
    user_pred_df.sort_values("pred_score", ascending=False, inplace=True)
    user_pred_df = user_pred_df.head(22)
    user_pred_df["rank"] = range(len(user_pred_df))
    
    return user_pred_df

In [ ]:
predictions = pd.concat([generate_predictions_for_user(user) for user in test_df["user_id"].unique()])

In [ ]:
# nDCGを計算
ndcg_scores = []
for user in predictions["user_id"].unique():
    true_scores = train_df.loc[train_df["user_id"] == user, "rank"].values
    pred_scores = predictions.loc[predictions["user_id"] == user, "pred_score"].values
    
    ndcg_scores.append(ndcg_at_k(true_scores, pred_scores))

print(f"Mean nDCG@22: {np.mean(ndcg_scores):.4f}")

In [ ]:
# エンコードしたカテゴリ変数の戻し
predictions["user_id"] = le_user.inverse_transform(predictions["user_id"])
predictions["product_id"] = le_product.inverse_transform(predictions["product_id"])
predictions_drop = predictions.drop('pred_score', axis=1)

In [ ]:
# 結果をファイルに出力
predictions_drop.to_csv('./predict/predictions_A.tsv', sep='\t', index=False)